# Task 2 — CLM-8B trên Kaggle (Hướng 2, chạy đầy đủ)

1. Settings: Accelerator **GPU T4 x2**, Internet **On**.
2. Add-ons → Secrets: `PUBLIC_TEST_DRIVE_ID` (file id Google Drive của bộ dữ liệu, xin từ nhóm), tuỳ chọn `GITHUB_TOKEN` nếu repo private, `HF_TOKEN` nếu cần.
3. Tuỳ chọn: thêm Kaggle dataset chứa `silver_labels.json` (output của `scripts/run_task2_api.py` ở chế độ predict) và sửa `SILVER_PATH` bên dưới.
4. Run All. Kết quả: `/kaggle/working/task2_predictions.json` và bảng benchmark.

In [ ]:
REPO = 'github.com/tiendinhquang2104/llm-final-project.git'
BRANCH = 'task-2'
SILVER_PATH = ''  # e.g. '/kaggle/input/task2-silver/silver_labels.json'; '' = gold only

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
def secret(name):
    try:
        return secrets.get_secret(name)
    except Exception:
        return ''
token = secret('GITHUB_TOKEN')
url = f'https://{token}@{REPO}' if token else f'https://{REPO}'
if not os.path.exists('/kaggle/working/repo'):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, '/kaggle/working/repo'], check=True)
os.chdir('/kaggle/working/repo')
if secret('HF_TOKEN'):
    os.environ['HF_TOKEN'] = secret('HF_TOKEN')
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q gdown 'transformers>=4.51' accelerate safetensors scikit-learn pyyaml bitsandbytes

In [ ]:
import gdown
drive_id = secret('PUBLIC_TEST_DRIVE_ID')
assert drive_id, 'Add the PUBLIC_TEST_DRIVE_ID secret'
os.makedirs('/kaggle/working/download', exist_ok=True)
path = gdown.download(id=drive_id, output='/kaggle/working/download/data.zip', quiet=False)
!python scripts/unpack_data.py /kaggle/working/download --dest data/raw
!ls data/raw

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
args = ['--config', 'configs/task2/clm_kaggle.yaml'] + (['--silver', SILVER_PATH] if SILVER_PATH else [])
subprocess.run(['python', 'scripts/run_task2_clm.py', *args], check=True)

In [ ]:
!python scripts/task2_benchmark.py
!cp outputs/task2/task2_clm_kaggle/predictions.json /kaggle/working/task2_predictions.json
import json
from src.task2.evaluator import validate_predictions
sub = json.load(open('/kaggle/working/task2_predictions.json'))
print(len(sub), validate_predictions(sub))